In [7]:
import json
import os


# =====================================================
# INPUT
# =====================================================

CEMETERY_FILE = "sg_cemeteries.geojson"
TEMPLE_FILE = "sg_main_god.geojson"
SCHOOL_POINT_FILE = "sg_chinese_schools_points.geojson"
SCHOOL_POLY_FILE = "sg_chinese_schools_polylines.geojson"

OUTPUT_DIR = "output"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =====================================================
# HELPERS
# =====================================================

def parse_popup_info(text):

    result = {
        "pinyin": None,
        "address": None,
        "country": None,
        "opened": None
    }

    if not text:
        return result

    for line in text.split("<br>"):

        if ":" not in line:
            continue

        key, value = line.split(":", 1)

        key = key.strip()
        value = value.strip()

        if key == "拼音":
            result["pinyin"] = value

        elif key == "Address R3":
            result["address"] = value

        elif key == "Country":
            result["country"] = value

        elif key == "创办时间公元 Opened Date":
            result["opened"] = value

    return result


# =====================================================
# CEMETERIES
# =====================================================

with open(CEMETERY_FILE, "r", encoding="utf-8") as f:
    cem_data = json.load(f)

cemeteries = []

for feat in cem_data["features"]:

    props = feat["properties"]
    lon, lat = feat["geometry"]["coordinates"]

    cemeteries.append({
        "id": f"cem-{props['OBJECTID']}",
        "type": "cemetery",

        "name": props.get("name"),

        "X": lon,
        "Y": lat,

        "mapyear": props.get("mapyear"),
        "yearfirst": props.get("yearfirst"),
        "yearlast": props.get("yearlast"),

        "category": props.get("category"),
        "cemid": props.get("cemid"),
        "comments": props.get("comments")
    })

with open(
    os.path.join(OUTPUT_DIR, "cemeteries.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(cemeteries, f, ensure_ascii=False, indent=2)


# =====================================================
# TEMPLES
# =====================================================

with open(TEMPLE_FILE, "r", encoding="utf-8") as f:
    temple_data = json.load(f)

temples = []

for feat in temple_data["features"]:

    props = feat["properties"]

    temples.append({
        "id": f"temple-{props['TempleID']}",
        "type": "temple",

        "SCNTemple": props.get("SCNTemple"),
        "TCNTemple": props.get("TCNTemple"),
        "ENTemple": props.get("ENTemple"),

        "Main_God_Ch": props.get("Main_God_Ch"),
        "Main_God_English": props.get("Main_God_English"),

        "Address": props.get("Address"),
        "Postal_Code": props.get("Postal_Code"),

        "X": props.get("LONGITUDE"),
        "Y": props.get("LATITUDE")
    })

with open(
    os.path.join(OUTPUT_DIR, "temples.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(temples, f, ensure_ascii=False, indent=2)


# =====================================================
# SCHOOLS
# =====================================================

with open(SCHOOL_POINT_FILE, "r", encoding="utf-8") as f:
    point_data = json.load(f)

with open(SCHOOL_POLY_FILE, "r", encoding="utf-8") as f:
    poly_data = json.load(f)

poly_lookup = {}

for feat in poly_data["features"]:

    name = feat["properties"]["Name"]

    coords = feat["geometry"]["coordinates"]

    polygon = []

    if coords and len(coords) > 0:

        ring = coords[0]

        polygon = [
            [p[0], p[1]]
            for p in ring
        ]

    poly_lookup[name] = polygon

schools = []

for feat in point_data["features"]:

    props = feat["properties"]

    lon = feat["geometry"]["coordinates"][0]
    lat = feat["geometry"]["coordinates"][1]

    popup = parse_popup_info(
        props.get("PopupInfo", "")
    )

    schools.append({
        "id": f"school-{props['OID']}",
        "type": "school",

        "Name": props.get("Name"),

        "pinyin": popup["pinyin"],
        "address": popup["address"],
        "country": popup["country"],
        "opened": popup["opened"],

        "X": lon,
        "Y": lat,

        "polygon": poly_lookup.get(
            props["Name"],
            []
        )
    })

with open(
    os.path.join(OUTPUT_DIR, "schools.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(schools, f, ensure_ascii=False, indent=2)

print("Done.")

Done.
